# 04 - LLM weak labelling of the unannotated symptoms (novelty N3)

Eleven of the 22 unified symptoms have **no gold labels in any open corpus**,
mostly the GAD-7 anxiety items. This notebook labels them with an open
instruction-tuned LLM and attaches a calibrated confidence to each label, so the
concept loss can be weighted instead of treating noisy labels as certain.

## The compute decision

Labelling all 1,017,959 weak-pool posts is not feasible: at a realistic 8
posts/second that is roughly 35 GPU-hours, far beyond a Colab session. So:

| | Plan |
|---|---|
| **Weak labels** | stratified **50,000**-post subsample, balanced across disorder and period |
| **Full 1M pool** | kept for self-supervised encoder pretraining, which needs no LLM |
| **Paper claim** | a learning curve over subsample size, which is stronger evidence than one large run |

## How weak-label quality is established

The weak-only symptoms have no gold labels, so their accuracy cannot be measured
directly. Two things close that gap:

1. **Section 5** runs the same labeller over the symptoms that *do* have gold
   labels and reports per-symptom F1, Cohen's kappa and calibration. That is the
   evidence the labeller is trustworthy at all, and it fits the confidence
   scaler.
2. **Section 8** emits an annotation sheet for a small in-house anxiety gold set,
   which validates the weak-only symptoms directly.

Everything is **resumable**: progress is appended to JSONL after every batch, so
a disconnected session loses at most one batch. Re-run the cell to continue.

## 1. Mount Drive and configure

In [ ]:
import os, sys
from google.colab import drive

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
DRIVE_PROCESSED = f'{DRIVE_BASE}/processed'
DRIVE_WEAK = f'{DRIVE_BASE}/weak_labels'
LOCAL_PROCESSED, LOCAL_WEAK = '/content/processed', '/content/weak_labels'

# How many posts to weak-label. 50,000 is roughly 2 hours on a T4.
N_SUBSAMPLE = 50_000
# Posts used to validate the labeller against gold symptoms.
N_VALIDATION = 1_500
# True exercises the whole pipeline in seconds with a fake labeller.
# Always do one mock pass before spending GPU time.
MOCK = True

# The label log is appended to local disk for speed, then snapshotted onto Drive
# every MIRROR_EVERY batches. /content is wiped when the runtime is recycled, so
# without the snapshot a dropped session loses the entire run, not one batch.
# 20 batches is ~640 posts, i.e. at most ~7 minutes of GPU time at risk.
MIRROR_EVERY = 20

# The labeller model is PINNED, not chosen from available GPU memory. Resume is
# keyed on (post_id, labeller): a session that landed on a smaller GPU and picked
# a different model would treat the existing log as foreign, archive it, and
# relabel 50,000 posts from scratch. Change this only deliberately.
LABELLER_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

os.environ.update(DRIVE_PROCESSED=DRIVE_PROCESSED, DRIVE_WEAK=DRIVE_WEAK,
                  LOCAL_PROCESSED=LOCAL_PROCESSED, LOCAL_WEAK=LOCAL_WEAK)
assert os.path.isdir(DRIVE_PROCESSED), f'{DRIVE_PROCESSED} not found - run notebook 02'
!mkdir -p $LOCAL_PROCESSED $LOCAL_WEAK "$DRIVE_WEAK"
!rsync -ah "$DRIVE_PROCESSED/" "$LOCAL_PROCESSED/"
# resume from whatever a previous session already wrote
!rsync -ah --exclude='*.partial' "$DRIVE_WEAK/" "$LOCAL_WEAK/"
print(sorted(os.listdir(LOCAL_PROCESSED)), '|', sorted(os.listdir(LOCAL_WEAK)))
for name in ('weak_labels.jsonl', 'validation_labels.jsonl'):
    f = f'{DRIVE_WEAK}/{name}'
    if os.path.exists(f):
        with open(f, 'rb') as fh:
            print(f'{name}: {sum(1 for _ in fh):,} rows already on Drive')

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q $REPO $CHECKOUT

need = ['src/symptograph/weak_label.py']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
assert not missing, f'stale checkout, missing {missing} - run: !rm -rf {CHECKOUT}'
sys.path.insert(0, f'{CHECKOUT}/src')
!git -C $CHECKOUT log --oneline -1

## 2. Which symptoms need weak labels

In [ ]:
import pandas as pd, numpy as np, json
from symptograph.preprocess import Ontology
from symptograph import weak_label as W

onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
WEAK_ONLY = W.weak_only_symptoms(onto)
GOLD = [s for s in onto.symptoms if s not in WEAK_ONLY]

print(f'{len(WEAK_ONLY)} symptoms need weak labels:')
for s in WEAK_ONLY:
    meta = next(m for m in onto.meta if m['name'] == s)
    print(f"  {s:24s} {meta['instrument']:6s} {meta['definition']}")
print(f'\n{len(GOLD)} symptoms have gold labels and are used for validation:')
print(' ', ', '.join(GOLD))

## 3. Stratified subsample

Controls are 57% of the pool but carry little symptom signal, so their share is
capped at 15%. The rest is spread evenly across the clinical disorder groups,
which keeps rare conditions such as bipolar and schizophrenia represented
instead of being swamped by r/depression. Sampling is seeded, so the subsample
is reproducible.

In [ ]:
sample = W.build_subsample(f'{LOCAL_PROCESSED}/weak_pool.parquet',
                           n=N_SUBSAMPLE, split='train', seed=0)
print(f'{len(sample):,} posts, all from the train split: '
      f"{set(sample.split) == {'train'}}\n")
display(sample.disorder_weak.value_counts().rename('posts').to_frame())
print('periods:', sample.period.value_counts().to_dict())
print('median words:', int(sample.text.str.split().map(len).median()))
sample.to_parquet(f'{LOCAL_WEAK}/subsample.parquet', index=False)

## 4. Load the labeller

The model is picked to match the GPU. Qwen2.5 is Apache-2.0 and not gated, so
nothing needs a Hugging Face token.

| GPU | Model |
|---|---|
| A100 / L4 (24 GB+) | `Qwen2.5-7B-Instruct` in fp16 |
| T4 (16 GB) | `Qwen2.5-7B-Instruct-AWQ`, 4-bit |
| no GPU | mock labeller only |

Keep `MOCK = True` for the first pass. It runs the entire pipeline, including
resume and calibration, in seconds.

In [ ]:
if MOCK:
    labeller = W.MockLabeller()
    print('mock labeller - no GPU used')
else:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    !pip install -q vllm 2>&1 | tail -2
    import torch
    assert torch.cuda.is_available(), 'no GPU - set MOCK=True or change runtime type'
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    # Fail loudly rather than silently switching models: a different model name
    # invalidates the resume log (see LABELLER_MODEL in section 1).
    if total_gb < 22 and 'AWQ' not in LABELLER_MODEL:
        raise RuntimeError(
            f'{total_gb:.0f} GB GPU cannot hold {LABELLER_MODEL} in fp16.\n'
            f'Get an L4/A100, or set LABELLER_MODEL to '
            f"{W.DEFAULT_MODELS['quantised']!r} - but note that switching model "
            'changes the labeller identity and discards existing progress.')
    print(f'{total_gb:.0f} GB GPU -> {LABELLER_MODEL}')
    labeller = W.VLLMLabeller(model=LABELLER_MODEL).load()
print('labeller:', labeller.name)

## 5. Is the labeller trustworthy?

Runs it over the symptoms that **do** have gold labels and compares. Weak labels
are only worth using if agreement here is respectable, and this is also what
fits the confidence scaler used in Section 7.

Judge the result on **Cohen's kappa** rather than accuracy, since several
symptoms are imbalanced.

In [ ]:
lab = pd.read_parquet(f'{LOCAL_PROCESSED}/labelled.parquet')
gold_cols = [onto.col(s) for s in GOLD]
val_pool = lab[(lab.split == 'train') & (lab[gold_cols] >= 0).any(axis=1)]
val_pool = val_pool.sample(n=min(N_VALIDATION, len(val_pool)), random_state=0)
gold_df = val_pool[['post_id', 'text'] + gold_cols].reset_index(drop=True)
print(f'{len(gold_df):,} gold posts for validation')

val_stats = W.run_labelling(gold_df[['post_id', 'text']], labeller, GOLD,
                            f'{LOCAL_WEAK}/validation_labels.jsonl', onto,
                            batch_size=32, verbose=True,
                            mirror_to=f'{DRIVE_WEAK}/validation_labels.jsonl',
                            mirror_every=MIRROR_EVERY)
print(val_stats)

In [ ]:
val_labels = W.load_labels(f'{LOCAL_WEAK}/validation_labels.jsonl', GOLD)
validation = W.validate_against_gold(val_labels, gold_df, GOLD, onto)
display(validation)
if not validation.empty:
    print(f"mean F1 {validation.f1.mean():.3f} | mean kappa {validation.cohen_kappa.mean():.3f}")
    if MOCK:
        print('(mock labeller - these scores are meaningless, only the shape matters)')

### Confidence calibration

Raw verbalised confidence from an LLM is usually overconfident. A single
temperature is fitted against observed accuracy on the gold symptoms, applied as
`p ** t`. One parameter keeps it stable on a small gold set, and it is monotone,
so the ranking of labels by confidence is preserved.

In [ ]:
temperature, ece_before, ece_after, reliability = W.fit_confidence_scaler(
    val_labels, gold_df, GOLD, onto)
print(f'temperature {temperature} | ECE {ece_before:.4f} -> {ece_after:.4f}')
display(reliability)

## 6. Label the subsample

The main run, over the 11 weak-only symptoms. Each batch is appended to JSONL
and the log is snapshotted to Drive every `MIRROR_EVERY` batches, so **if the
session drops, just re-run this cell** - it recovers the snapshot and continues
from where it stopped. The snapshot is also taken on the way out of an
interrupt, so stopping the cell by hand is safe too.

On an L4 expect roughly 5 hours for 50,000 posts, longer if posts are long.
Section 9 copies the remaining artefacts to Drive; the label log itself is
already there.

In [ ]:
run_stats = W.run_labelling(sample, labeller, WEAK_ONLY,
                            f'{LOCAL_WEAK}/weak_labels.jsonl', onto,
                            batch_size=32, verbose=True, report_every=20,
                            mirror_to=f'{DRIVE_WEAK}/weak_labels.jsonl',
                            mirror_every=MIRROR_EVERY)
print()
print(json.dumps(run_stats, indent=2))

## 7. Apply calibration and save

In [ ]:
weak = W.load_labels(f'{LOCAL_WEAK}/weak_labels.jsonl', WEAK_ONLY)
print(f'{len(weak):,} posts labelled\n')

rates = pd.DataFrame({
    'positive_rate': [round(float((weak[f'wl_{s}'] == 1).mean()), 3) for s in WEAK_ONLY],
    'unparsed': [int((weak[f'wl_{s}'] < 0).sum()) for s in WEAK_ONLY],
    'mean_confidence': [round(float(weak[f'conf_{s}'].mean()), 3) for s in WEAK_ONLY],
}, index=WEAK_ONLY).sort_values('positive_rate', ascending=False)
display(rates)

paths = W.save_weak_labels(weak, sample, LOCAL_WEAK, WEAK_ONLY,
                           stats=run_stats, temperature=temperature,
                           validation=validation)
print({k: f'{os.path.getsize(v) / 1e6:.2f} MB' for k, v in paths.items()})

## 8. Annotation sheet for the in-house anxiety gold set

The weak-only symptoms still have no gold labels, so weak-label quality on them
cannot be measured directly. This writes 800 posts (600 from the anxiety
subreddits, 200 controls) with one blank column per weak-only symptom.

Have **two annotators label it independently**, report Cohen's kappa per item,
and resolve disagreements. That is roughly 15-20 hours each, and it gives both a
direct validation set and a releasable resource contribution for the paper.

In [ ]:
ann = W.build_annotation_sample(f'{LOCAL_PROCESSED}/weak_pool.parquet',
                                f'{LOCAL_WEAK}/annotation_sample.csv',
                                n=800, control_n=200, seed=0)
print(f'{len(ann)} posts ->  annotation_sample.csv')
print('subreddits:', ann.subreddit.value_counts().head(6).to_dict())
print('\ncolumns to fill in:', [c for c in ann.columns if c in WEAK_ONLY
                                 or c in ('evidence_spans', 'annotator')])

## 9. Copy results to Drive

The label logs are mirrored to Drive during the run already; this copies the
derived artefacts (parquet, stats, annotation sheet). Safe to run on a partial
pass.

In [ ]:
!mkdir -p "$DRIVE_WEAK"
!rsync -ah --info=progress2 --exclude='*.partial' "$LOCAL_WEAK/" "$DRIVE_WEAK/"
!ls -lh "$DRIVE_WEAK"

## Notes

**Using the labels.** `weak_labels.parquet` carries `wl_<symptom>` (1 present,
0 absent, -1 unparsed) and `conf_<symptom>` in `[0, 1]`, already calibrated.
Weight the concept loss by confidence, and **never treat -1 as a negative** -
mask it out, exactly as with the `-1` entries from notebook 02.

**Option 3, the full pool.** The remaining ~968,000 unlabelled posts stay useful
without any LLM: masked-language-model pretraining of the encoder on in-domain
text, which typically helps more than extra weak labels. Read them lazily:

```python
import pyarrow.parquet as pq
f = pq.ParquetFile(f'{DRIVE_BASE}/processed/weak_pool.parquet')
for batch in f.iter_batches(batch_size=10_000, columns=['post_id', 'text']):
    ...
```

**Ablation this enables (N3).** Train the concept bottleneck on 10%, 25%, 50%
and 100% of the weak labels and plot symptom F1 against weak-label volume,
separately for the gold-supervised and weak-only symptoms. That curve is the
evidence for the N3 claim that calibrated weak supervision substitutes for
scarce expert annotation.

**Reminder.** Run the labeller locally as configured here. Sending these posts
to a third-party API would breach the data agreements for several of the
corpora.

**Next:** notebook 05 trains the concept bottleneck (N1) over the graph (N2),
using the gold labels plus these confidence-weighted weak labels (N3).